# Structured Outputs
This notebook demonstrates how to work with different output formats and parsers when working with Language Models (LLM).

## What we'll learn:
- Basic string output parsing
- Working with tools and structured outputs
- Using Pydantic models for type validation
- Understanding different parser types and their use cases

### Setup

In [ ]:

import os
from datetime import datetime
from pydantic import BaseModel, Field
from lib.agents import Agent
from lib.state_machine import Run
from typing import List,Any, Annotated
from lib.llm import LLM
from tavily import TavilyClient
from typing import Dict
from lib.tooling import tool
from lib.messages import UserMessage, SystemMessage,BaseMessage
from dotenv import load_dotenv
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG

## Setup

In [91]:
load_dotenv()

True

In [93]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Load data into Vector DB

In [94]:
db = VectorStoreManager(OPENAI_API_KEY)
db


VectorStoreManager():<chromadb.api.client.Client object at 0x0000026269EDDA70>

In [95]:
loader_service = CorpusLoaderService(db)

In [96]:
rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

In [97]:
games_market_rag = RAG(
    llm=rag_llm,
    vector_store = loader_service.load_pdf(
        store_name="games_market",
        pdf_path="TheGamingIndustry2024.pdf",
    )
)

VectorStore `games_market` ready!
Pages from `TheGamingIndustry2024.pdf` added!


In [98]:
result:Run = games_market_rag.invoke("What's the  state of virtual reality")
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
The state of virtual reality (VR) in 2024 is characterized by its significant impact on the gaming industry, allowing players to immerse themselves in realistic and interactive digital environments. VR technology is empowering creators to develop engaging gaming experiences, as exemplified by games like "Half-Life: Alyx," which offers high-fidelity graphics, intuitive controls, and immersive audio. Additionally, VR is being utilized beyond entertainment, with applications in education, physical rehabilitation, and mental health, showcasing its transformative potential in various fields. Overall, VR is evolving as a key technology that enhances player engagement and reshapes the gaming landscape.


## Tools

In [99]:
@tool
def retrieve_game(query):
    """
    Search the vector database for relevant information.
    
    Source: The Gaming Industry in 2024 - Trends, Technologies & Predictions
    Publisher: Ixie Gaming
    Release: 2024
    ```
        The gaming industry, on the brink of transformative change due to technological
        advancements, is redefining entertainment and social interaction with
        immersive, personalized, and interactive experiences.
    ```

    args:
        query (str): Search query
    """
    result:Run = games_market_rag.invoke(query)
    return result.get_final_state()["answer"]

In [100]:
@tool
def evaluate_retrieval(num_games: int = 1, top: bool = True) -> List[Dict]:
    """
    Returns the top or bottom N games with highest or lowest scores.    
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    data = [
        {"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98},
        {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97},
        {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97},
        {"Game": "Super Smash Bros. Brawl", "Platform": "Wii", "Score": 93},
        {"Game": "Mario Kart 8 Deluxe", "Platform": "Switch", "Score": 92},
        {"Game": "Fire Emblem: Awakening", "Platform": "3DS", "Score": 92},
        {"Game": "Donkey Kong Country Returns", "Platform": "Wii", "Score": 87},
        {"Game": "Luigi's Mansion 3", "Platform": "Switch", "Score": 86},
        {"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85},
        {"Game": "Animal Crossing: New Leaf", "Platform": "3DS", "Score": 88}
    ]
    # Sort the games list by Score
    sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)
    return sorted_games[:num_games]

In [101]:

@tool
def game_web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    
    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    
    return formatted_results

In [103]:
class GamesInformation(BaseModel):
    """A formatted information summary of a game"""
    title: Annotated[str, Field(description="Title of the game")]   
    developer: Annotated[str, Field(description="Developer of the game")]
    release_date: Annotated[str, Field(description="Release date of the game")]
    platforms: Annotated[List[str], Field(description="List of platforms the game is available on")]
    description: Annotated[str, Field(description="Brief description of the game")]
    genres: Annotated[List[str], Field(description="List of genres the game belongs to")]
    publisher: Annotated[List[str], Field(description="List of publishers of the game")]

In [ ]:
class StructuredAgent:
    """An AI Agent that provides structured outputs"""
    
    def __init__(
        self,
        role: str = "Meeting Assistant",
        instructions: str = "Help summarize meetings and track action items",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[Any] = None,
        output_model: BaseModel = None
    ):
        """Initialize the agent with its configuration
        
        Args:
            role: The agent's role/persona
            instructions: Basic instructions for the agent
            model: The LLM model to use
            temperature: Creativity parameter (0.0 = more deterministic)
            tools: List of tools the agent can use
            output_model: Pydantic model for structured output
        """
        self.model = model
        self.role = role
        self.instructions = instructions
        self.tools = tools
        self.output_model = output_model

        # Load environment variables
        load_dotenv()
        
        # Initialize the LLM
        self.llm = LLM(
            model=model,
            temperature=temperature,
            tools=tools,
        )

    def invoke(self, user_message: str) -> dict:
        """Process a user message and return a structured response
        
        Args:
            user_message: The user's input message
            
        Returns:
            A dictionary containing the structured response
        """
        messages = [
            SystemMessage(
                content=(
                    f"You're an AI Agent and your role is {self.role}. "  
                    f"Your instructions: {self.instructions}"
                )
            )
        ]
        # Add user message
        messages.append(UserMessage(content=user_message))
        
        # Get AI response with structured format
        if self.output_model:
            ai_message = self.llm.invoke(input=messages, response_format=self.output_model)
            parser = JsonOutputParser()
            return parser.parse(ai_message)
        else:
            # Handle unstructured response if no model specified
            ai_message = self.llm.invoke(messages)
            return {"response": ai_message.content}


In [106]:
agentic_rag = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game,evaluate_retrieval,game_web_search],
    instructions=(
        "You are an Agentic RAG assistant that can intelligently decide which tools to use "        
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not."
        "Perform a web search if the retrieved documents are not enough to answer the question, and use the retrieved information to decide the best query to perform the web search."
    ),
    output_model=GamesInformation
)

TypeError: Agent.__init__() got an unexpected keyword argument 'output_model'

In [88]:
def print_messages(messages: List[BaseMessage]):
    for m in messages: 
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

In [ ]:
run_1 = agentic_rag.invoke(
    query= (
        "Who developed FIFA 21?"
    ),
    session_id="games",
)

print("\nMessages from run 1:")
messages = run_1.get_final_state()["messages"]
print_messages(messages)

In [ ]:
run_2 = agentic_rag.invoke(
    query= (
        "When was God of War Ragnarok released?"
    ),
    session_id="games",
)

print("\nMessages from run 2:")
messages = run_2.get_final_state()["messages"]
print_messages(messages)

In [ ]:
run_3 = agentic_rag.invoke(
    query= (
        "What platform was Pokémon Red launched on?"
    ),
    session_id="games",
)

print("\nMessages from run 3:")
messages = run_3.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an Agentic RAG assistant that can intelligently decide which tools to use Your task is to evaluate if the documents are enough to respond the query. Give a detailed explanation, so it's possible to take an action to accept it or not.Perform a web search if the retrieved documents are not enough to answer the question, and use the retrieved information to decide the best query to perform the web search., tool_calls = None)
 -> (role = user, content = How games are revolutionizing education?, tool_calls = None)
 -> (role = assistant, content = To determine if the available documents provide enough information to respond to the query about how games are revolutionizing educati